In [ ]:
#x-z磁場のストリームラインon密度map

import os
import re
import glob
from collections import defaultdict

import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt

from scipy.interpolate import griddata
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter


# ============================================================
# 1. ユーザー設定
# ============================================================

# VTKファイルがあるディレクトリ
vtk_dir = os.path.expanduser(
    "~/athena-project/results/〇〇"
)

# 全体図の保存先
output_dir1 = os.path.join(
    vtk_dir,
    "xz_density_with_magnetic_fieldlines"
)

# 1/10ズーム図の保存先
output_dir2 = os.path.join(
    vtk_dir,
    "xz_density_with_magnetic_fieldlines_zoom_1over10"
)

# 1/100ズーム図の保存先
output_dir3 = os.path.join(
    vtk_dir,
    "xz_density_with_magnetic_fieldlines_zoom_1over100"
)

os.makedirs(output_dir1, exist_ok=True)
os.makedirs(output_dir2, exist_ok=True)
os.makedirs(output_dir3, exist_ok=True)

# zoomの各軸幅
zoom_fraction_10 = 1.0 / 10.0
zoom_fraction_100 = 1.0 / 100.0

# 補間後の画像解像度
plot_resolution = 700

# 磁力線の密度
stream_density = 1.5

# 図のサイズ
figsize = (9, 8)

# 保存画像のdpi
dpi = 200

print("=== Output directories ===")
print(f"Full images : {output_dir1}")
print(f"1/10 zoom   : {output_dir2}")
print(f"1/100 zoom  : {output_dir3}")


# ============================================================
# 2. Toyouchi.cppのコード単位
# ============================================================

Munit = 4.0e33       # g
Lunit = 6.7e15       # cm
Tunit = 3.34e10      # s

Vunit = Lunit / Tunit
Rhounit = Munit / Lunit**3
Punit = Rhounit * Vunit**2

# Athena++の磁場単位 [Gauss]
Bunit = np.sqrt(4.0 * np.pi * Punit)

AU = 1.496e13
YEAR = 3.15576e7

print("\n=== Code units ===")
print(f"Lunit   = {Lunit:.6e} cm")
print(f"Rhounit = {Rhounit:.6e} g/cm^3")
print(f"Bunit   = {Bunit:.6e} Gauss")
print(f"Tunit   = {Tunit/(1.0e6*YEAR):.6e} Myr")


# ============================================================
# 3. VTKファイルを出力時刻ごとに整理
# ============================================================

timestep_dict = defaultdict(list)

vtk_pattern = os.path.join(
    vtk_dir,
    "Toyouchi.block*.out2.*.vtk"
)

vtk_files = glob.glob(vtk_pattern)

if not vtk_files:
    raise FileNotFoundError(
        "VTK files were not found.\n"
        f"Checked pattern:\n{vtk_pattern}"
    )

for filename in vtk_files:
    match = re.search(
        r"\.out2\.(\d+)\.vtk$",
        filename
    )

    if match:
        step = int(match.group(1))
        timestep_dict[step].append(filename)

timesteps = sorted(timestep_dict)

if not timesteps:
    raise RuntimeError(
        "Could not extract timestep numbers "
        "from VTK filenames."
    )

# block番号順に並べる
for step in timesteps:
    timestep_dict[step] = sorted(
        timestep_dict[step],
        key=lambda filename: int(
            re.search(
                r"block(\d+)",
                filename
            ).group(1)
        )
    )

print("\n=== VTK information ===")
print(f"Number of VTK files     : {len(vtk_files)}")
print(f"Number of output steps  : {len(timesteps)}")
print(f"First step              : {timesteps[0]:05d}")
print(f"Last step               : {timesteps[-1]:05d}")
print(
    f"Blocks at first step    : "
    f"{len(timestep_dict[timesteps[0]])}"
)
print(
    f"Blocks at last step     : "
    f"{len(timestep_dict[timesteps[-1]])}"
)


# ============================================================
# 4. Athena++ VTKヘッダーから時刻を読む
# ============================================================

def read_athena_vtk_time(filename):
    """
    Athena++ legacy VTKの2行目からコード時刻を取得する。
    """

    with open(filename, "rb") as file:
        file.readline()
        title = file.readline().decode(
            "ascii",
            errors="ignore"
        )

    match = re.search(
        r"time=([0-9eE+\-.]+)",
        title
    )

    if match is None:
        return np.nan

    return float(match.group(1))


# ============================================================
# 5. 配列名を探す
# ============================================================

def find_array_name(grid, candidates):
    """
    candidatesの中から、VTKに存在する配列名を返す。
    """

    for name in candidates:
        if name in grid.array_names:
            return name

    raise KeyError(
        "Required array was not found.\n"
        f"Candidates: {candidates}\n"
        f"Available arrays: {grid.array_names}"
    )


# ============================================================
# 6. x-z断面を読み込む
# ============================================================

def load_xz_slice(step):
    """
    y=0面を含むMeshBlockから、
    y=0に最も近いセル層を抽出する。

    偶数セル数の場合、y=0を挟む上下2層が選ばれる。
    後で同じ(x,z)座標について平均する。
    """

    x_list = []
    z_list = []
    rho_list = []
    B_list = []

    # y=0と交差する全MeshBlockの実境界
    domain_xmin = np.inf
    domain_xmax = -np.inf
    domain_zmin = np.inf
    domain_zmax = -np.inf

    files = timestep_dict[step]

    time_code = read_athena_vtk_time(
        files[0]
    )

    for filename in files:
        grid = pv.read(filename)

        # PyVista bounds:
        # xmin, xmax, ymin, ymax, zmin, zmax
        bounds = grid.bounds

        ymin = bounds[2]
        ymax = bounds[3]

        # このMeshBlockがy=0面を含まなければ除外
        if not (ymin <= 0.0 <= ymax):
            continue

        domain_xmin = min(domain_xmin, bounds[0])
        domain_xmax = max(domain_xmax, bounds[1])
        domain_zmin = min(domain_zmin, bounds[4])
        domain_zmax = max(domain_zmax, bounds[5])

        points = grid.cell_centers().points

        rho_name = find_array_name(
            grid,
            [
                "rho",
                "dens",
                "density",
                "prim_dens",
                "prim_density",
            ]
        )

        B_name = find_array_name(
            grid,
            [
                "Bcc",
                "bcc",
                "B",
                "magnetic_field",
            ]
        )

        rho = np.asarray(
            grid[rho_name]
        ).reshape(-1)

        B = np.asarray(
            grid[B_name]
        ).reshape(-1, 3)

        y = points[:, 1]

        # このMeshBlockでy=0に最も近い距離
        min_abs_y = np.min(
            np.abs(y)
        )

        # 丸め誤差を許容
        tolerance = max(
            1.0e-10,
            min_abs_y * 1.0e-6
        )

        mask = (
            np.abs(
                np.abs(y) - min_abs_y
            )
            <= tolerance
        )

        if not np.any(mask):
            continue

        x_list.append(
            points[mask, 0]
        )

        z_list.append(
            points[mask, 2]
        )

        rho_list.append(
            rho[mask]
        )

        B_list.append(
            B[mask]
        )

    if not x_list:
        raise RuntimeError(
            "No cells near y=0 were found.\n"
            f"step = {step:05d}"
        )

    return {
        "step": step,
        "time_code": time_code,
        "x_code": np.concatenate(x_list),
        "z_code": np.concatenate(z_list),
        "rho_code": np.concatenate(rho_list),
        "B_code": np.vstack(B_list),
        "domain_xmin_code": domain_xmin,
        "domain_xmax_code": domain_xmax,
        "domain_zmin_code": domain_zmin,
        "domain_zmax_code": domain_zmax,
    }


# ============================================================
# 7. 同じx-z座標の値を平均する
# ============================================================

def average_duplicate_xz(
    x,
    z,
    rho,
    B
):
    """
    同じ(x,z)座標に複数のセル値がある場合に平均する。

    y=0を挟む2層やMeshBlock境界の重複を処理する。
    """

    coords = np.column_stack([
        x,
        z
    ])

    unique_coords, inverse = np.unique(
        coords,
        axis=0,
        return_inverse=True
    )

    counts = np.bincount(
        inverse
    ).astype(float)

    rho_sum = np.bincount(
        inverse,
        weights=rho
    )

    Bx_sum = np.bincount(
        inverse,
        weights=B[:, 0]
    )

    By_sum = np.bincount(
        inverse,
        weights=B[:, 1]
    )

    Bz_sum = np.bincount(
        inverse,
        weights=B[:, 2]
    )

    rho_avg = rho_sum / counts

    B_avg = np.column_stack([
        Bx_sum / counts,
        By_sum / counts,
        Bz_sum / counts
    ])

    return (
        unique_coords[:, 0],
        unique_coords[:, 1],
        rho_avg,
        B_avg
    )


# ============================================================
# 8. 全出力で共通の密度カラースケールを決める
# ============================================================

print(
    "\n[INFO] Determining common "
    "density color scale..."
)

density_samples = []

for n, step in enumerate(timesteps):
    slice_data = load_xz_slice(step)

    rho_cgs = (
        slice_data["rho_code"]
        * Rhounit
    )

    valid = (
        np.isfinite(rho_cgs)
        & (rho_cgs > 0.0)
    )

    if np.any(valid):
        density_samples.append(
            rho_cgs[valid]
        )

    print(
        f"  [{n+1:3d}/{len(timesteps):3d}] "
        f"step={step:05d}"
    )

if not density_samples:
    raise RuntimeError(
        "No positive finite density values "
        "were found."
    )

all_slice_density = np.concatenate(
    density_samples
)

rho_vmin = np.percentile(
    all_slice_density,
    0.5
)

# 全時刻の最大密度を含める
rho_vmax = np.max(
    all_slice_density
)

if not np.isfinite(rho_vmin):
    raise RuntimeError(
        "rho_vmin is not finite."
    )

if not np.isfinite(rho_vmax):
    raise RuntimeError(
        "rho_vmax is not finite."
    )

if rho_vmin <= 0.0:
    raise RuntimeError(
        "rho_vmin must be positive "
        "for LogNorm."
    )

if rho_vmax <= rho_vmin:
    raise RuntimeError(
        "Invalid density color range:\n"
        f"rho_vmin={rho_vmin}\n"
        f"rho_vmax={rho_vmax}"
    )

print(
    f"\nrho_vmin = {rho_vmin:.6e} g/cm^3"
)

print(
    f"rho_vmax = {rho_vmax:.6e} g/cm^3"
)


# ============================================================
# 9. 1時刻分を描画・保存する
# ============================================================

def plot_xz_density_with_fieldlines(
    step,
    output_dir,
    zoom_fraction=None,
    filename_suffix=""
):
    """
    x-z密度マップへ、x-z面に射影した磁力線を重ねる。

    Parameters
    ----------
    step : int
        out2番号

    output_dir : str
        保存先

    zoom_fraction : float or None
        Noneなら実データ全体。0.01なら各軸幅を全体の1/100にする。

    filename_suffix : str
        出力ファイル名に付ける文字列
    """

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    data = load_xz_slice(step)

    x_code = data["x_code"]
    z_code = data["z_code"]
    rho_code = data["rho_code"]
    B_code = data["B_code"]

    # 重複するx-z座標を平均
    (
        x_code,
        z_code,
        rho_code,
        B_code
    ) = average_duplicate_xz(
        x_code,
        z_code,
        rho_code,
        B_code
    )

    # --------------------------------------------------------
    # コード単位からCGS・表示単位へ変換
    # --------------------------------------------------------

    x_AU = (
        x_code
        * Lunit
        / AU
    )

    z_AU = (
        z_code
        * Lunit
        / AU
    )

    rho_cgs = (
        rho_code
        * Rhounit
    )

    Bx_microG = (
        B_code[:, 0]
        * Bunit
        * 1.0e6
    )

    By_microG = (
        B_code[:, 1]
        * Bunit
        * 1.0e6
    )

    Bz_microG = (
        B_code[:, 2]
        * Bunit
        * 1.0e6
    )

    Bmag_microG = np.sqrt(
        Bx_microG**2
        + By_microG**2
        + Bz_microG**2
    )

    time_code = data["time_code"]

    time_myr = (
        time_code
        * Tunit
        / (1.0e6 * YEAR)
    )

    # --------------------------------------------------------
    # 描画範囲
    # --------------------------------------------------------

    # セル中心の最小・最大ではなく、VTKブロック境界から実領域を取得
    data_xmin = data["domain_xmin_code"] * Lunit / AU
    data_xmax = data["domain_xmax_code"] * Lunit / AU
    data_zmin = data["domain_zmin_code"] * Lunit / AU
    data_zmax = data["domain_zmax_code"] * Lunit / AU

    if zoom_fraction is None:
        xmin, xmax = data_xmin, data_xmax
        zmin, zmax = data_zmin, data_zmax
        range_label = "full domain"
    else:
        if not (0.0 < zoom_fraction <= 1.0):
            raise ValueError(
                "zoom_fraction must satisfy 0 < zoom_fraction <= 1"
            )

        x_center = 0.0 if data_xmin <= 0.0 <= data_xmax else 0.5 * (data_xmin + data_xmax)
        z_center = 0.0 if data_zmin <= 0.0 <= data_zmax else 0.5 * (data_zmin + data_zmax)

        x_half = 0.5 * (data_xmax - data_xmin) * zoom_fraction
        z_half = 0.5 * (data_zmax - data_zmin) * zoom_fraction

        xmin = max(data_xmin, x_center - x_half)
        xmax = min(data_xmax, x_center + x_half)
        zmin = max(data_zmin, z_center - z_half)
        zmax = min(data_zmax, z_center + z_half)
        range_label = f"zoom = {zoom_fraction:.3g} of full width"

    print(
        f"[RANGE] step={step:05d}, {range_label}\n"
        f"        full x={data_xmin:.6e} -- {data_xmax:.6e} AU\n"
        f"        full z={data_zmin:.6e} -- {data_zmax:.6e} AU\n"
        f"        plot x={xmin:.6e} -- {xmax:.6e} AU\n"
        f"        plot z={zmin:.6e} -- {zmax:.6e} AU"
    )

    if xmax <= xmin:
        raise ValueError(
            f"Invalid x range: {xmin}, {xmax}"
        )

    if zmax <= zmin:
        raise ValueError(
            f"Invalid z range: {zmin}, {zmax}"
        )

    # 表示範囲の外側を少し含めて補間する
    margin_x = 0.05 * (
        xmax - xmin
    )

    margin_z = 0.05 * (
        zmax - zmin
    )

    mask = (
        (x_AU >= xmin - margin_x)
        & (x_AU <= xmax + margin_x)
        & (z_AU >= zmin - margin_z)
        & (z_AU <= zmax + margin_z)
        & np.isfinite(rho_cgs)
        & (rho_cgs > 0.0)
        & np.isfinite(Bx_microG)
        & np.isfinite(Bz_microG)
    )

    number_of_source_points = (
        np.count_nonzero(mask)
    )

    if number_of_source_points < 10:
        raise RuntimeError(
            "Too few source cells in the "
            "requested plotting range.\n"
            f"step={step:05d}\n"
            f"source cells={number_of_source_points}\n"
            f"x range={xmin:.3e} to {xmax:.3e} AU\n"
            f"z range={zmin:.3e} to {zmax:.3e} AU"
        )

    # --------------------------------------------------------
    # 規則格子
    # --------------------------------------------------------

    x_grid = np.linspace(
        xmin,
        xmax,
        plot_resolution
    )

    z_grid = np.linspace(
        zmin,
        zmax,
        plot_resolution
    )

    X, Z = np.meshgrid(
        x_grid,
        z_grid
    )

    source_points = np.column_stack([
        x_AU[mask],
        z_AU[mask]
    ])

    # --------------------------------------------------------
    # 密度をlog空間で補間
    # --------------------------------------------------------

    log_rho_source = np.log10(
        rho_cgs[mask]
    )

    log_rho_linear = griddata(
        source_points,
        log_rho_source,
        (X, Z),
        method="linear"
    )

    log_rho_nearest = griddata(
        source_points,
        log_rho_source,
        (X, Z),
        method="nearest"
    )

    log_rho_grid = np.where(
        np.isfinite(log_rho_linear),
        log_rho_linear,
        log_rho_nearest
    )

    rho_grid = (
        10.0**log_rho_grid
    )

    # --------------------------------------------------------
    # Bx, Bzを補間
    # --------------------------------------------------------

    Bx_linear = griddata(
        source_points,
        Bx_microG[mask],
        (X, Z),
        method="linear"
    )

    Bz_linear = griddata(
        source_points,
        Bz_microG[mask],
        (X, Z),
        method="linear"
    )

    Bx_nearest = griddata(
        source_points,
        Bx_microG[mask],
        (X, Z),
        method="nearest"
    )

    Bz_nearest = griddata(
        source_points,
        Bz_microG[mask],
        (X, Z),
        method="nearest"
    )

    Bx_grid = np.where(
        np.isfinite(Bx_linear),
        Bx_linear,
        Bx_nearest
    )

    Bz_grid = np.where(
        np.isfinite(Bz_linear),
        Bz_linear,
        Bz_nearest
    )

    Bx_grid = np.nan_to_num(
        Bx_grid,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    Bz_grid = np.nan_to_num(
        Bz_grid,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    # streamplotへゼロ磁場だけを渡さない
    Bproj_grid = np.sqrt(
        Bx_grid**2
        + Bz_grid**2
    )

    if not np.any(
        Bproj_grid > 0.0
    ):
        raise RuntimeError(
            "Projected magnetic field is zero "
            f"at step={step:05d}."
        )

    # --------------------------------------------------------
    # 描画
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=figsize
    )

    density_map = ax.pcolormesh(
        X,
        Z,
        rho_grid,
        # 論文図風：低密度=淡い黄緑、高密度=濃い青
        cmap="YlGnBu",
        norm=LogNorm(
            vmin=rho_vmin,
            vmax=rho_vmax
        ),
        shading="auto",
        rasterized=True
    )

    ax.streamplot(
        x_grid,
        z_grid,
        Bx_grid,
        Bz_grid,
        color="black",
        density=stream_density,
        linewidth=0.65,
        arrowsize=0.8,
        arrowstyle="->",
        minlength=0.02,
        maxlength=20.0,
        integration_direction="both",
        # 他の磁力線に近づいても積分を打ち切らない
        broken_streamlines=False
    )

    colorbar = fig.colorbar(
        density_map,
        ax=ax,
        pad=0.02
    )

    colorbar.set_label(
        r"Density [g cm$^{-3}$]"
    )

    # --------------------------------------------------------
    # 表示範囲に応じて、目盛を10^n AU単位で表示
    # 例：表示範囲が10^5 AU級なら x [10^5 AU]
    # --------------------------------------------------------
    maximum_absolute_coordinate = max(
        abs(xmin),
        abs(xmax),
        abs(zmin),
        abs(zmax),
    )

    if maximum_absolute_coordinate > 0.0:
        scale_exponent = int(
            np.floor(
                np.log10(maximum_absolute_coordinate)
            )
        )
    else:
        scale_exponent = 0

    axis_scale_AU = 10.0**scale_exponent

    axis_formatter = FuncFormatter(
        lambda value, position: (
            f"{value / axis_scale_AU:g}"
        )
    )

    ax.xaxis.set_major_formatter(
        axis_formatter
    )
    ax.yaxis.set_major_formatter(
        axis_formatter
    )

    ax.set_xlabel(
        rf"$x\ [10^{{{scale_exponent}}}\ {{\rm AU}}]$"
    )
    ax.set_ylabel(
        rf"$z\ [10^{{{scale_exponent}}}\ {{\rm AU}}]$"
    )

    ax.set_title(
        "Density and projected magnetic field lines\n"
        f"step={step:05d}, "
        f"t={time_myr:.3e} Myr"
    )

    ax.set_xlim(
        xmin,
        xmax
    )

    ax.set_ylim(
        zmin,
        zmax
    )

    ax.set_aspect(
        "equal",
        adjustable="box"
    )

    ax.grid(
        True,
        alpha=0.15,
        linestyle="--"
    )

    # 表示範囲内の磁場最大値
    Bmax_visible = np.max(
        Bmag_microG[mask]
    )

    Bxmax_visible = np.max(
        np.abs(Bx_microG[mask])
    )

    Bzmax_visible = np.max(
        np.abs(Bz_microG[mask])
    )

    info = (
        f"$B_{{\\rm max}}$ = "
        f"{Bmax_visible:.3e} $\\mu$G\n"
        f"$|B_x|_{{\\rm max}}$ = "
        f"{Bxmax_visible:.3e} $\\mu$G\n"
        f"$|B_z|_{{\\rm max}}$ = "
        f"{Bzmax_visible:.3e} $\\mu$G\n"
        f"source cells = "
        f"{number_of_source_points}"
    )

    ax.text(
        0.02,
        0.98,
        info,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9,
        color="white",
        bbox={
            "boxstyle": "round",
            "facecolor": "black",
            "alpha": 0.55,
            "edgecolor": "white",
        }
    )

    plt.tight_layout()

    if filename_suffix:
        suffix = (
            "_" + filename_suffix
        )
    else:
        suffix = ""

    output_file = os.path.join(
        output_dir,
        (
            f"xz_density_Blines_"
            f"{step:05d}"
            f"{suffix}.png"
        )
    )

    fig.savefig(
        output_file,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white"
    )

    # メモリ解放
    plt.close(fig)

    print(
        f"[SAVED] step={step:05d}, "
        f"time={time_myr:.3e} Myr\n"
        f"        {output_file}"
    )

    return output_file


# ============================================================
# 10. 全体図をoutput_dir1へ保存
# ============================================================

print("\n=== Generating full-domain images ===")

full_output_files = []

for n, step in enumerate(timesteps):
    print(
        f"[FULL {n+1:3d}/{len(timesteps):3d}] "
        f"Processing step={step:05d}"
    )

    output_file = (
        plot_xz_density_with_fieldlines(
            step=step,
            output_dir=output_dir1,
            zoom_fraction=None,
            filename_suffix="full"
        )
    )

    full_output_files.append(
        output_file
    )


# ============================================================
# 11. 1/10ズーム図をoutput_dir2へ保存
# ============================================================

print("\n=== Generating 1/10 zoom images ===")

zoom10_output_files = []

for n, step in enumerate(timesteps):
    print(
        f"[ZOOM 1/10 {n+1:3d}/{len(timesteps):3d}] "
        f"Processing step={step:05d}"
    )

    output_file = (
        plot_xz_density_with_fieldlines(
            step=step,
            output_dir=output_dir2,
            zoom_fraction=zoom_fraction_10,
            filename_suffix="zoom_1over10"
        )
    )

    zoom10_output_files.append(
        output_file
    )


# ============================================================
# 12. 1/100ズーム図をoutput_dir3へ保存
# ============================================================

print("\n=== Generating 1/100 zoom images ===")

zoom100_output_files = []

for n, step in enumerate(timesteps):
    print(
        f"[ZOOM 1/100 {n+1:3d}/{len(timesteps):3d}] "
        f"Processing step={step:05d}"
    )

    output_file = (
        plot_xz_density_with_fieldlines(
            step=step,
            output_dir=output_dir3,
            zoom_fraction=zoom_fraction_100,
            filename_suffix="zoom_1over100"
        )
    )

    zoom100_output_files.append(
        output_file
    )


# ============================================================
# 13. 保存結果を確認
# ============================================================

print("\n=== Finished ===")
print(
    f"Full images : "
    f"{len(full_output_files)}"
)

print(
    f"1/10 zoom   : "
    f"{len(zoom10_output_files)}"
)

print(
    f"1/100 zoom  : "
    f"{len(zoom100_output_files)}"
)

print(f"\nFull output directory:\n{output_dir1}")
print(f"\n1/10 zoom output directory:\n{output_dir2}")
print(f"\n1/100 zoom output directory:\n{output_dir3}")

print("\nFull images:")
for filename in full_output_files:
    print(
        "  ",
        os.path.basename(filename)
    )

print("\n1/10 zoom images:")
for filename in zoom10_output_files:
    print(
        "  ",
        os.path.basename(filename)
    )

print("\n1/100 zoom images:")
for filename in zoom100_output_files:
    print(
        "  ",
        os.path.basename(filename)
    )
